In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import xgboost as xgb

In [ ]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')

print(train.shape)

In [ ]:
cols_to_drop = train.columns[train.isnull().mean() > 0.9]
train = train.drop(columns=cols_to_drop)

for col in train.columns:
    if train[col].dtype == 'object':
        train[col] = train[col].fillna('missing')
    else:
        train[col] = train[col].fillna(-999)

In [ ]:
new_features = pd.DataFrame(index=train.index)

new_features['TransactionAmt_log'] = np.log1p(train['TransactionAmt'])
new_features['TransactionDT_hours'] = train['TransactionDT'] / 3600
new_features['TransactionDT_days'] = train['TransactionDT'] / (3600 * 24)

new_features['P_emaildomain_prefix'] = train['P_emaildomain'].astype(str).str.split('.').str[0]

for col in ['card1', 'card2', 'card3', 'card5']:
    freq = train[col].value_counts()
    new_features[col + '_freq'] = train[col].map(freq)

train = pd.concat([train, new_features], axis=1)
train = train.copy()

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_cols = train.select_dtypes(include='object').columns

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))

In [ ]:
X = train.drop(columns=['isFraud', 'TransactionID'])
y = train['isFraud']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    use_label_encoder=False,
    tree_method='hist'  
)

model.fit(X_train, y_train)

In [ ]:
y_pred_train = model.predict_proba(X_train)[:, 1]
y_pred_val = model.predict_proba(X_val)[:, 1]

roc_train = roc_auc_score(y_train, y_pred_train)
roc_val = roc_auc_score(y_val, y_pred_val)

print("Train ROC:", roc_train)
print("Val ROC:", roc_val)

In [2]:
!pip install mlflow dagshub

In [3]:
import dagshub
dagshub.init(repo_owner='slosa23', repo_name='ML-Assignment2', mlflow=True)

Accessing as slosa23

Initialized MLflow to track repo "slosa23/ML-Assignment2"

Repository slosa23/ML-Assignment2 initialized!

In [ ]:
import mlflow

mlflow.set_experiment("XGBoost_Training")

with mlflow.start_run(run_name="XGB_Baseline"):

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)

    mlflow.log_metric("roc_auc_train", roc_train)
    mlflow.log_metric("roc_auc_val", roc_val)

    mlflow.xgboost.log_model(model, "model")

    print("XGBoost baseline logged")

In [ ]:
depths = [4, 6, 8]

for depth in depths:
    
    with mlflow.start_run(run_name=f"XGB_depth_{depth}"):

        model = xgb.XGBClassifier(
            n_estimators=200,
            max_depth=depth,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='auc',
            tree_method='hist'
        )

        model.fit(X_train, y_train)

        y_pred = model.predict_proba(X_val)[:, 1]
        roc = roc_auc_score(y_val, y_pred)

        mlflow.log_param("max_depth", depth)
        mlflow.log_metric("roc_auc_val", roc)

        print(f"depth={depth} | ROC={roc:.4f}")

In [ ]:
lrs = [0.05, 0.1, 0.2]

for lr in lrs:
    
    with mlflow.start_run(run_name=f"XGB_lr_{lr}"):

        model = xgb.XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=lr,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='auc',
            tree_method='hist'
        )

        model.fit(X_train, y_train)

        y_pred = model.predict_proba(X_val)[:, 1]
        roc = roc_auc_score(y_val, y_pred)

        mlflow.log_param("learning_rate", lr)
        mlflow.log_metric("roc_auc_val", roc)

        print(f"lr={lr} | ROC={roc:.4f}")

In [ ]:
estimators = [100, 200, 400]

for n in estimators:
    
    with mlflow.start_run(run_name=f"XGB_estimators_{n}"):

        model = xgb.XGBClassifier(
            n_estimators=n,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='auc',
            tree_method='hist'
        )

        model.fit(X_train, y_train)

        y_pred = model.predict_proba(X_val)[:, 1]
        roc = roc_auc_score(y_val, y_pred)

        mlflow.log_param("n_estimators", n)
        mlflow.log_metric("roc_auc_val", roc)

        print(f"n_estimators={n} | ROC={roc:.4f}")

In [ ]:
best_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    tree_method='hist'
)

best_model.fit(X_train, y_train)

In [ ]:
y_pred_train = best_model.predict_proba(X_train)[:, 1]
y_pred_val = best_model.predict_proba(X_val)[:, 1]

roc_train = roc_auc_score(y_train, y_pred_train)
roc_val = roc_auc_score(y_val, y_pred_val)

print("Train ROC:", roc_train)
print("Val ROC:", roc_val)

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("model", best_model)
])

In [ ]:
with mlflow.start_run(run_name="XGB_BEST"):

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 400)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.2)

    mlflow.log_metric("roc_auc_train", roc_train)
    mlflow.log_metric("roc_auc_val", roc_val)

    mlflow.sklearn.log_model(pipeline, name="model")

    print("Best XGBoost model logged.")

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
import numpy as np

class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.cols_to_drop = None
        self.cat_cols = None
        self.freq_maps = {}
        self.label_encoders = {}

    def fit(self, X, y=None):
        X = X.copy()

        self.cols_to_drop = X.columns[X.isnull().mean() > 0.9]
        X = X.drop(columns=self.cols_to_drop)

        self.cat_cols = X.select_dtypes(include='object').columns

        for col in ['card1', 'card2', 'card3', 'card5']:
            if col in X.columns:
                self.freq_maps[col] = X[col].value_counts()

        from sklearn.preprocessing import LabelEncoder
        for col in self.cat_cols:
            le = LabelEncoder()
            le.fit(X[col].astype(str).fillna('missing'))
            self.label_encoders[col] = le

        return self

    def transform(self, X):
        X = X.copy()

        X = X.drop(columns=self.cols_to_drop, errors='ignore')

        for col in X.columns:
            if col in self.cat_cols:
                X[col] = X[col].fillna('missing')
            else:
                X[col] = X[col].fillna(-999)

        if 'TransactionAmt' in X.columns:
            X['TransactionAmt_log'] = np.log1p(X['TransactionAmt'])

        if 'TransactionDT' in X.columns:
            X['TransactionDT_hours'] = X['TransactionDT'] / 3600
            X['TransactionDT_days'] = X['TransactionDT'] / (3600 * 24)

        if 'P_emaildomain' in X.columns:
            X['P_emaildomain_prefix'] = X['P_emaildomain'].astype(str).str.split('.').str[0]

        for col in ['card1', 'card2', 'card3', 'card5']:
            if col in X.columns and col in self.freq_maps:
                X[col + '_freq'] = X[col].map(self.freq_maps[col])

        for col in self.cat_cols:
            if col in X.columns:
                le = self.label_encoders[col]
                X[col] = le.transform(X[col].astype(str))

        return X

In [ ]:
train = train_transaction.merge(train_identity, on='TransactionID', how='left')

X_raw = train.drop(columns=['isFraud', 'TransactionID'])
y = train['isFraud']

In [ ]:
preprocessor = FraudPreprocessor()
preprocessor.fit(X_raw)

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", best_model)
])

In [ ]:
import mlflow

mlflow.set_experiment("XGBoost_Training")  

with mlflow.start_run(run_name="XGB_FINAL_PIPELINE"):

    mlflow.log_param("model", "XGBoost_pipeline")
    mlflow.log_param("n_estimators", 400)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.2)

    mlflow.log_metric("roc_auc_train", roc_train)
    mlflow.log_metric("roc_auc_val", roc_val)

    mlflow.sklearn.log_model(pipeline, artifact_path="model", registered_model_name="best-model")

    print("FINAL PIPELINE LOGGED")

In [4]:
import numpy as np
import pandas as pd
import xgboost as xgb
import mlflow

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [5]:
class FraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.cols_to_drop = None
        self.freq_maps = {}
        self.cat_cols = None
        self.encoder = None

    def _feature_engineering(self, X):
        X = X.copy()

        new_cols = {}

        if 'TransactionAmt' in X.columns:
            new_cols['TransactionAmt_log'] = np.log1p(X['TransactionAmt'])

        if 'TransactionDT' in X.columns:
            new_cols['TransactionDT_hours'] = X['TransactionDT'] / 3600
            new_cols['TransactionDT_days'] = X['TransactionDT'] / (3600 * 24)

        if 'P_emaildomain' in X.columns:
            new_cols['P_emaildomain_prefix'] = (
                X['P_emaildomain'].astype(str).str.split('.').str[0]
            )

        for col in ['card1', 'card2', 'card3', 'card5']:
            if col in X.columns and col in self.freq_maps:
                new_cols[col + '_freq'] = X[col].map(self.freq_maps[col]).fillna(0)

        X = pd.concat([X, pd.DataFrame(new_cols, index=X.index)], axis=1)

        return X

    def fit(self, X, y=None):
        X = X.copy()

        self.cols_to_drop = X.columns[X.isnull().mean() > 0.9]
        X = X.drop(columns=self.cols_to_drop)

        for col in ['card1', 'card2', 'card3', 'card5']:
            if col in X.columns:
                self.freq_maps[col] = X[col].value_counts()

        X = self._feature_engineering(X)

        for col in X.columns:
            if X[col].dtype == 'object':
                X[col] = X[col].fillna("missing")
            else:
                X[col] = X[col].fillna(-999)

        self.cat_cols = X.select_dtypes(include='object').columns.tolist()

        self.encoder = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )
        self.encoder.fit(X[self.cat_cols].astype(str))

        return self

    def transform(self, X):
        X = X.copy()

        X = X.drop(columns=self.cols_to_drop, errors='ignore')

        X = self._feature_engineering(X)

        for col in X.columns:
            if col in self.cat_cols:
                X[col] = X[col].fillna("missing")
            else:
                X[col] = X[col].fillna(-999)

        if len(self.cat_cols) > 0:
            X[self.cat_cols] = self.encoder.transform(
                X[self.cat_cols].astype(str)
            )

        return X

In [6]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

train = train_transaction.merge(train_identity, on='TransactionID', how='left')

X_raw = train.drop(columns=['isFraud', 'TransactionID'])
y = train['isFraud']


X_train, X_val, y_train, y_val = train_test_split(
    X_raw, y, test_size=0.2, random_state=42
)

In [7]:
model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.2,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    tree_method='hist'
)

In [8]:
pipeline = Pipeline([
    ("preprocess", FraudPreprocessor()),
    ("model", model)
])


pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocess', FraudPreprocessor()),
                ('model',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=0.8, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric='auc',
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.2,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=6, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=400, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [9]:
y_pred_train = pipeline.predict_proba(X_train)[:, 1]
y_pred_val = pipeline.predict_proba(X_val)[:, 1]

roc_train = roc_auc_score(y_train, y_pred_train)
roc_val = roc_auc_score(y_val, y_pred_val)

print("Train ROC:", roc_train)
print("Val ROC:", roc_val)

Train ROC: 0.9888817606117727
Val ROC: 0.9650174982971498


In [11]:
mlflow.set_experiment("XGBoost_Training")

with mlflow.start_run(run_name="XGB_FINAL_PIPELINE_FIXED"):

    mlflow.log_param("model", "XGBoost_pipeline")
    mlflow.log_param("n_estimators", 400)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.2)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)

    mlflow.log_metric("roc_auc_train", roc_train)
    mlflow.log_metric("roc_auc_val", roc_val)

    mlflow.sklearn.log_model(
        pipeline,
        artifact_path="model",
        registered_model_name="xgb-best-model"
    )

    print("FINAL PIPELINE LOGGED")

2026/05/04 23:24:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 23:24:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'xgb-best-model'.
2026/05/04 23:25:08 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: xgb-best-model, version 1
Created version '1' of model 'xgb-best-model'.


FINAL PIPELINE LOGGED
🏃 View run XGB_FINAL_PIPELINE_FIXED at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1/runs/a70b1c872b2c4f3c88afa26ff9ddac7b
🧪 View experiment at: https://dagshub.com/slosa23/ML-Assignment2.mlflow/#/experiments/1
